In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "../data/test_datasets/mbd_dataset/detail/trx/fold=0"
)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7634228 entries, 0 to 7634227
Data columns (total 14 columns):
 #   Column         Dtype         
---  ------         -----         
 0   client_id      object        
 1   event_time     datetime64[ns]
 2   amount         float32       
 3   event_type     int32         
 4   event_subtype  int32         
 5   currency       float64       
 6   src_type11     float64       
 7   src_type12     float64       
 8   dst_type11     float64       
 9   dst_type12     float64       
 10  src_type21     float64       
 11  src_type22     float64       
 12  src_type31     float64       
 13  src_type32     float64       
dtypes: datetime64[ns](1), float32(1), float64(9), int32(2), object(1)
memory usage: 728.1+ MB


In [3]:
df["year_month"] = df["event_time"].dt.to_period("M")
# для учёта сезонности потом
df["month"] = df["year_month"].dt.month

In [4]:
df.describe()

,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32,month
count,7634228,7.634228e+06,7.634228e+06,7.634228e+06,7.634192e+06,7.588765e+06,7.588765e+06,7.580026e+06,7.580026e+06,7.626199e+06,7.626199e+06,7.618069e+06,7.618069e+06,7.634228e+06
mean,2022-01-06 06:06:45.856158720,7.037684e+05,2.249315e+01,2.753563e+01,1.099737e+01,3.467387e+01,2.243373e+02,7.358390e+02,2.077095e+04,2.454554e+04,3.627590e+01,1.190242e+03,4.038951e+01,6.679041e+00
min,2020-12-31 21:00:11.062887,1.168688e-05,1.000000e+00,1.000000e+00,1.000000e+00,4.000000e+00,1.000000e+00,5.000000e+00,2.120000e+02,1.600000e+01,1.000000e+00,3.000000e+00,1.000000e+00,1.000000e+00
25%,2021-07-06 18:29:05.728897536,2.297636e+03,1.000000e+00,1.200000e+01,1.100000e+01,2.200000e+01,4.700000e+01,3.060000e+02,1.606700e+04,1.299300e+04,9.000000e+00,4.940000e+02,1.900000e+01,4.000000e+00
50%,2022-01-02 08:07:45.843692800,1.666204e+04,1.300000e+01,2.300000e+01,1.100000e+01,2.200000e+01,4.700000e+01,7.800000e+02,2.161900e+04,2.502600e+04,2.900000e+01,1.119000e+03,3.600000e+01,7.000000e+00
75%,2022-07-10 16:12:07.381889536,8.850691e+04,4.600000e+01,4.000000e+01,1.100000e+01,2.200000e+01,4.650000e+02,7.930000e+02,2.676600e+04,3.716800e+04,5.600000e+01,1.846000e+03,6.000000e+01,1.000000e+01
max,2022-12-31 20:59:56.338108,8.811039e+10,5.600000e+01,6.200000e+01,1.700000e+01,1.850000e+02,1.140000e+03,1.604000e+03,3.305500e+04,4.892300e+04,8.800000e+01,2.506000e+03,8.900000e+01,1.200000e+01
std,NaN,6.397719e+07,2.126929e+01,1.645170e+01,2.060438e-01,3.372444e+01,3.017901e+02,4.893076e+02,7.106693e+03,1.397403e+04,2.714266e+01,7.541996e+02,2.304687e+01,3.431670e+00


In [5]:
df_11 = df[df["currency"] == 11]

client_month_features = (
    df.groupby(["client_id", "year_month"])
      .agg(
          tx_total=("amount", "count")     # общее количество транзакций
      )
      .join(
          df_11.groupby(["client_id", "year_month"]).agg(
              amount_11_sum=("amount", "sum"),
              amount_11_mean=("amount", "mean"),
              amount_11_median=("amount", "median"),
              amount_11_std=("amount", "std"),
              amount_11_min=("amount", "min"),
              amount_11_max=("amount", "max"),
              tx_11_count=("amount", "count")
          )
      )
      .reset_index()
)

In [6]:
import pandas as pd
import numpy as np

# Список всех клиентов и месяцев 
all_clients = client_month_features["client_id"].unique()
all_months = pd.period_range(client_month_features["year_month"].min(),
                             client_month_features["year_month"].max(),
                             freq="M")

# Полная сетка client × month 
full_index = pd.MultiIndex.from_product([all_clients, all_months],
                                        names=["client_id", "year_month"])

# Сетку объединяем с агрегированными данными 
client_month_features = (
    client_month_features
        .set_index(["client_id", "year_month"])
        .reindex(full_index)  # добавляет пропущенные client-month
        .reset_index()
)

# Заполняем NaN нулями для всех признаков
fill_cols = ["tx_total", "amount_11_sum", "amount_11_mean", "amount_11_median",
             "amount_11_std", "amount_11_min", "amount_11_max", "tx_11_count"]

client_month_features[fill_cols] = client_month_features[fill_cols].fillna(0)

# Проверка 
print(client_month_features.groupby("client_id").size().describe())  # сколько месяцев на клиента

count    20032.0
mean        25.0
std          0.0
min         25.0
25%         25.0
50%         25.0
75%         25.0
max         25.0
dtype: float64


In [7]:
client_month_features.describe()

,tx_total,amount_11_sum,amount_11_mean,amount_11_median,amount_11_std,amount_11_min,amount_11_max,tx_11_count
count,500800.000000,5.008000e+05,5.008000e+05,5.008000e+05,5.008000e+05,5.008000e+05,5.008000e+05,500800.000000
mean,15.244065,1.053583e+07,3.795899e+05,9.521756e+04,8.756281e+05,3.990694e+04,4.333100e+06,15.228946
std,58.098190,4.657933e+08,1.788113e+07,5.914776e+06,5.030252e+07,5.743855e+06,2.178209e+08,58.073196
min,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
50%,5.000000,2.627758e+05,3.597411e+04,9.371892e+03,3.544339e+04,7.778734e+01,1.226297e+05,5.000000
75%,20.000000,2.235519e+06,1.355353e+05,3.788883e+04,2.120976e+05,2.095875e+03,7.982909e+05,20.000000
max,8301.000000,1.187547e+11,7.404870e+09,3.964256e+09,2.541650e+10,3.964256e+09,8.811039e+10,8301.000000


In [8]:
# Сколько месяцев у каждого клиента
client_month_features.groupby("client_id")["year_month"].nunique().describe()

count    20032.0
mean        25.0
std          0.0
min         25.0
25%         25.0
50%         25.0
75%         25.0
max         25.0
Name: year_month, dtype: float64

In [9]:
client_month_features = client_month_features.sort_values(["client_id", "year_month"])
for lag in [1, 2, 3, 6, 9, 12]:
    client_month_features[f"lag{lag}_amount_11_sum"] = client_month_features.groupby("client_id")["amount_11_sum"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_mean"] = client_month_features.groupby("client_id")["amount_11_mean"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_median"] = client_month_features.groupby("client_id")["amount_11_median"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_std"] = client_month_features.groupby("client_id")["amount_11_std"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_min"] = client_month_features.groupby("client_id")["amount_11_min"].shift(lag)
    client_month_features[f"lag{lag}_tx_11_count"] = client_month_features.groupby("client_id")["tx_11_count"].shift(lag)
    client_month_features[f"lag{lag}_tx_total"] = client_month_features.groupby("client_id")["tx_total"].shift(lag)

for window in [3, 6, 12]:
    # Сумма
    client_month_features[f"amount_11_sum_rolling_mean_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    
    # Количество транзакций
    client_month_features[f"tx_11_count_rolling_mean_{window}m"] = (
        client_month_features.groupby("client_id")["tx_11_count"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    
    # Общее количество транзакций
    client_month_features[f"tx_total_rolling_mean_{window}m"] = (
        client_month_features.groupby("client_id")["tx_total"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    
    # Максимум за период
    client_month_features[f"amount_11_sum_rolling_max_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).max())
    )
    
    # Минимум за период
    client_month_features[f"amount_11_sum_rolling_min_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).min())
    )
    
    # Стандартное отклонение (волатильность)
    client_month_features[f"amount_11_sum_rolling_std_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )

In [10]:
# Накопленная сумма (вся история до текущего месяца)
client_month_features["amount_11_cumsum"] = (
    client_month_features
        .groupby("client_id")["amount_11_sum"]
        .apply(lambda x: x.cumsum().shift(1))
        .reset_index(level=0, drop=True)
)

# Накопленное количество транзакций в валюте 11
client_month_features["tx_11_cumsum"] = (
    client_month_features
        .groupby("client_id")["tx_11_count"]
        .apply(lambda x: x.cumsum().shift(1))
        .reset_index(level=0, drop=True)
)

# Накопленное общее количество транзакций
client_month_features["tx_total_cumsum"] = (
    client_month_features
        .groupby("client_id")["tx_total"]
        .apply(lambda x: x.cumsum().shift(1))
        .reset_index(level=0, drop=True)
)


# Средний чек накопленным итогом (до текущего месяца)
client_month_features["amount_11_cummean"] = (
    client_month_features["amount_11_cumsum"] / 
    client_month_features["tx_11_cumsum"].replace(0, np.nan)
)

In [11]:
# до текущего месяца

# Максимум за всю историю (до текущего месяца)
client_month_features["amount_11_hist_max"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().max())
)

# Минимум за всю историю
client_month_features["amount_11_hist_min"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().min())
)

# Среднее за всю историю
client_month_features["amount_11_hist_mean"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Стандартное отклонение за всю историю
client_month_features["amount_11_hist_std"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().std())
)
# Доля транзакций в валюте 11 накопленным итогом
client_month_features["tx_11_share_cum"] = (
    client_month_features["tx_11_cumsum"] / 
    client_month_features["tx_total_cumsum"]
)

In [12]:
# Извлекаем месяц и квартал
client_month_features["month"] = client_month_features["year_month"].dt.month
client_month_features["quarter"] = client_month_features["year_month"].dt.quarter

# Среднее за этот же месяц в прошлом году (если данных достаточно)
client_month_features["amount_11_same_month_last_year"] = (
    client_month_features
        .groupby("client_id")["amount_11_sum"]
        .shift(12)
)

# Флаг начала квартала
client_month_features["is_quarter_start"] = (client_month_features["month"] % 3 == 1).astype(int)

# Флаг конца квартала
client_month_features["is_quarter_end"] = (client_month_features["month"] % 3 == 0).astype(int)


In [13]:
# Безопасные динамические признаки через лаги

# Используем lag1_amount_11_sum и lag2_amount_11_sum
# lag1 — сумма за прошлый месяц
# lag2 — сумма за месяц до прошлого (т.е. два месяца назад)

# Изменение относительно прошлого месяца (процентное)
client_month_features["amount_11_pct_change_lag"] = (
    (client_month_features["lag1_amount_11_sum"] - client_month_features["lag2_amount_11_sum"])
    / client_month_features["lag2_amount_11_sum"].replace(0, np.nan) * 100
)

# Изменение относительно прошлого месяца (абсолютное)
client_month_features["amount_11_abs_change_lag"] = (
    client_month_features["lag1_amount_11_sum"] - client_month_features["lag2_amount_11_sum"]
)

# Отношение к среднему за последние 3 месяца (rolling3 уже сдвинуто на 1 месяц)
client_month_features["amount_11_vs_3m_avg_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_sum_rolling_mean_3m"].replace(0, np.nan)
)

# Отношение к среднему за всю историю (cumulative)
client_month_features["amount_11_vs_cummean_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_cummean"].replace(0, np.nan)
)

# Отношение к максимуму за всю историю
client_month_features["amount_11_vs_hist_max_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_hist_max"].replace(0, np.nan)
)

# Отношение к минимуму за всю историю
client_month_features["amount_11_vs_hist_min_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_hist_min"].replace(0, np.nan)
)

# Коэффициент вариации (CV = std/mean) — уже безопасен, так как hist_std и hist_mean строятся через shift(1)
client_month_features["amount_11_hist_cv_lag"] = (
    client_month_features["amount_11_hist_std"] / client_month_features["amount_11_hist_mean"].replace(0, np.nan)
)

# Размах (max - min)
client_month_features["amount_11_hist_range_lag"] = (
    client_month_features["amount_11_hist_max"] - client_month_features["amount_11_hist_min"]
)

# Был ли всплеск (превышение среднего на 2 сигмы)
client_month_features["amount_11_is_spike_lag"] = (
    (client_month_features["lag1_amount_11_sum"] > 
     client_month_features["amount_11_hist_mean"] + 2 * client_month_features["amount_11_hist_std"])
).astype(int)

# Был ли провал
client_month_features["amount_11_is_dip_lag"] = (
    (client_month_features["lag1_amount_11_sum"] < 
     client_month_features["amount_11_hist_mean"] - 2 * client_month_features["amount_11_hist_std"])
).astype(int)


# Количество месяцев подряд с положительной динамикой
def count_consecutive_positives_lag(x):
    """Считает, сколько месяцев подряд сумма росла,
       используем lag1 для текущей позиции, lag2 для сравнения"""
    result = []
    count = 0
    for i in range(len(x)):
        if i > 1 and x.iloc[i-1] > x.iloc[i-2]:  # сравниваем lag1 и lag2
            count += 1
        else:
            count = 0
        result.append(count)
    return pd.Series(result, index=x.index)

client_month_features["amount_11_consecutive_growth_lag"] = (
    client_month_features.groupby("client_id")["lag1_amount_11_sum"]
    .transform(lambda x: count_consecutive_positives_lag(x))
)


# Нулевые транзакции и доля активности
# Был ли месяц с нулевыми транзакциями (через lag1)
client_month_features["amount_11_is_zero_lag"] = (client_month_features["lag1_amount_11_sum"] == 0).astype(int)

# Доля месяцев с активностью (накопленным итогом), безопасно через shift(1)
client_month_features["amount_11_active_ratio_lag"] = (
    client_month_features.groupby("client_id")["amount_11_is_zero_lag"]
    .transform(lambda x: 1 - x.shift(1).expanding().mean())
)

In [14]:
client_month_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500800 entries, 0 to 500799
Data columns (total 97 columns):
 #   Column                            Non-Null Count   Dtype    
---  ------                            --------------   -----    
 0   client_id                         500800 non-null  object   
 1   year_month                        500800 non-null  period[M]
 2   tx_total                          500800 non-null  float64  
 3   amount_11_sum                     500800 non-null  float32  
 4   amount_11_mean                    500800 non-null  float32  
 5   amount_11_median                  500800 non-null  float64  
 6   amount_11_std                     500800 non-null  float64  
 7   amount_11_min                     500800 non-null  float32  
 8   amount_11_max                     500800 non-null  float32  
 9   tx_11_count                       500800 non-null  float64  
 10  lag1_amount_11_sum                480768 non-null  float32  
 11  lag1_amount_11_mean       

In [15]:
# Event type признаки (кумулятивно до текущего месяца)
# Считаем количество транзакций каждого типа по клиенту и месяцу
event_counts_monthly = (
    df.groupby(["client_id", "year_month", "event_type"])["amount"]
    .count()
    .unstack(fill_value=0)
)

# Чтобы получить кумулятивные признаки до месяца M, сдвигаем на 1 месяц
event_counts_cumsum = event_counts_monthly.groupby(level=0).cumsum().shift(fill_value=0)

# Считаем долю каждого типа от всех транзакций до текущего месяца
event_shares_cumsum = event_counts_cumsum.div(event_counts_cumsum.sum(axis=1), axis=0)

# Берем топ-10 типов событий, остальные объединяем в "other_events"
top_events_cumsum = event_shares_cumsum.iloc[:, :10].copy()
top_events_cumsum["other_events"] = 1 - top_events_cumsum.sum(axis=1)

# Сбрасываем индекс для объединения с другими признаками
event_features = top_events_cumsum.reset_index()


# src/dst признаки (кумулятивно до текущего месяца)
src_cols = ["src_type11", "src_type12", "src_type21", "src_type22", "src_type31", "src_type32"]
dst_cols = ["dst_type11", "dst_type12"]

# Базовая таблица с client_id и год-месяцем
src_dst_features = df[["client_id", "year_month"]].drop_duplicates().copy()

for col in src_cols + dst_cols:
    # Считаем уникальные значения по клиенту и месяцу
    stats = (
        df.groupby(["client_id", "year_month"])[col]
        .nunique()
        .groupby(level=0)       # группируем по клиенту
        .cumsum()               # кумулятивная сумма по месяцам
        .shift(fill_value=0)    # сдвигаем на 1 месяц, чтобы исключить текущий
        .rename(f"{col}_n_unique")
        .reset_index()
    )
    # Объединяем с базовой таблицей
    src_dst_features = src_dst_features.merge(stats, on=["client_id", "year_month"], how="left")


In [16]:
src_dst_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364817 entries, 0 to 364816
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype    
---  ------               --------------   -----    
 0   client_id            364817 non-null  object   
 1   year_month           364817 non-null  period[M]
 2   src_type11_n_unique  364817 non-null  int64    
 3   src_type12_n_unique  364817 non-null  int64    
 4   src_type21_n_unique  364817 non-null  int64    
 5   src_type22_n_unique  364817 non-null  int64    
 6   src_type31_n_unique  364817 non-null  int64    
 7   src_type32_n_unique  364817 non-null  int64    
 8   dst_type11_n_unique  364817 non-null  int64    
 9   dst_type12_n_unique  364817 non-null  int64    
dtypes: int64(8), object(1), period[M](1)
memory usage: 27.8+ MB


In [17]:
client_month_features = client_month_features.merge(src_dst_features, on=["client_id", "year_month"], how="left")

In [18]:
client_month_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500800 entries, 0 to 500799
Columns: 105 entries, client_id to dst_type12_n_unique
dtypes: float32(26), float64(69), int64(8), object(1), period[M](1)
memory usage: 351.5+ MB


In [19]:
# Сохраняем финальный датасет
version = 3
client_month_features.to_parquet(f"../data/processed/client_month_features_{version}.parquet", index=False)

print(f"Финальный датасет: {client_month_features.shape[0]} строк, {client_month_features.shape[1]} признаков")


Финальный датасет: 500800 строк, 105 признаков


In [ ]:
# не используем для обучения
current_month_features = [
    'amount_11_sum',      # то, что предсказываем
    'amount_11_mean',     # статистика этого месяца
    'amount_11_median',
    'amount_11_std',
    'amount_11_min',
    'amount_11_max',
    'tx_11_count',
    'tx_total'] 